In [ ]:
## Get needed functions
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from numpy import float32
import torch 
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
from torch.utils.data import t

In [3]:
## Import created functions
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))
from src.dataset import SingleCellDataset
from src.model import SingleCellModel
from torch.utils.data import DataLoader
from src.train import train
from src.evaluate import evaluate

Now we will download data, transform it into tensors, and run through the whole pipeline. I am going to use train_test_split from scikit because it lets me stratify the split by y names.

In [ ]:
adata = sc.datasets.pbmc68k_reduced()

## Get the gene expression counts as X and the cell labels as y
X_numpy = adata.X
y_series = adata.obs["bulk_labels"]

## Transform X to a tensor object. Convert y string labels to integers
X_tensor = torch.tensor(X_numpy, dtype = torch.float32)


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_series)
#print(label_encoder.classes_)
#print(y_encoded[:10])

## Now convert encoded y names to a tensor. we use torch.long to represent 64 bit (integers) vs floats
y_tensor = torch.tensor(y_encoded, dtype = torch.long)

## Define dataset
dataset = SingleCellDataset(X_tensor, y_tensor)

## Split into train, validation, and test using train_test_split. Ensure split is same always/

indices = list(range(len(dataset)))
labels = y_tensor.numpy()

train_idx, temporary_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=labels)

val_idx, test_idx = train_test_split(
    temporary_idx,
    test_size=0.50,
    random_state=42,
    stratify=labels[temporary_idx])

train_dataset = Subset(dataset, train_idx)
validate_dateset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)


## Now we get the train/validate/test dataloaders

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
validate_dataloader = DataLoader(validate_dateset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

Epoch 1/20 complete. Loss: 0.7269
Epoch 2/20 complete. Loss: 0.4107
Epoch 3/20 complete. Loss: 0.1775
Epoch 4/20 complete. Loss: 0.0217
Epoch 5/20 complete. Loss: 0.0166
Epoch 6/20 complete. Loss: 0.0149
Epoch 7/20 complete. Loss: 0.0051
Epoch 8/20 complete. Loss: 0.0033
Epoch 9/20 complete. Loss: 0.0016
Epoch 10/20 complete. Loss: 0.0037
Epoch 11/20 complete. Loss: 0.0014
Epoch 12/20 complete. Loss: 0.0029
Epoch 13/20 complete. Loss: 0.0002
Epoch 14/20 complete. Loss: 0.0100
Epoch 15/20 complete. Loss: 0.0026
Epoch 16/20 complete. Loss: 0.0012
Epoch 17/20 complete. Loss: 0.0002
Epoch 18/20 complete. Loss: 0.0002
Epoch 19/20 complete. Loss: 0.0018
Epoch 20/20 complete. Loss: 0.0006
accuracy: 0.8286 | average_loss: 0.6874


Now the goal is to tune hyperparameters using the train/validation set. I will use the optuna package to tune the hyperparaeters

In [ ]:
import optuna 

def objective(trial):
    ## Define hyperparameter space
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log = True)
    weight_decay =  trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    hidden_size1 = trial.suggest_float("hidden_size1", [64, 128, 256])
    hidden_size2 = trial.suggest_float("hidden_size2", [32, 64, 128])
    dropout_rate =  trial.suggest_float('dropout_rate', 0.1, 0.6)
    

    ## Call model 
    model = SingleCellModel(
    num_features=X_tensor.shape[1],
    num_classes=len(label_encoder.classes_),
    hidden_size1 = hidden_size1, 
    hidden_size2 = hidden_size2,
    dropout_rate = dropout_rate)

    ## Train model
    train(train_dataloader, model, num_epochs = 20, lr = lr, weight_decay = weight_decay)

    accuracy, average_loss = evaluate(test_dataloader, model)

    return average_loss
    
study = optuna.create_study()
study.optimize(objective, n_trials=100)

In [ ]:
## Redefine model

model = SingleCellModel(
    num_features=X_tensor.shape[1],
    num_classes=len(label_encoder.classes_)
)

## Train the model
history = train(train_dataloader, model, num_epochs = 20, lr = lr, weight_decay = weight_decay)

## Evaluate the model
accuracy, average_loss = evaluate(test_dataloader, model)
print(f"accuracy: {accuracy:.4f} | average_loss: {average_loss:.4f}")